# Week 3: Prescriptive Dynamic Pricing Optimization Engine
## Constrained Resource Allocation via SciPy & Causal Machine Learning

Welcome to the **Week 3 Prescriptive Optimization Engine** of the **EconoCausal** system.
In Weeks 1 and 2, we built the Causal DAG and trained an EconML Double Machine Learning (`LinearDML` + `LightGBM`) model to estimate the Individual Treatment Effect (ITE) $\tau_i(t)$ of discounts ($10 and $20) for each customer.

In Week 3, we transition from **Predictive / Causal Inference** to **Prescriptive Decision-Making**:
How do we strategically allocate a finite marketing budget (e.g., $5,000) across thousands of customers and multiple discount tiers to maximize incremental conversions?

---
### Mathematical Formulation: Multiple-Choice Knapsack Problem (MCKP)

Let:
- $N$ = total number of customers
- $x_{i, 10} \in \{0, 1\}$ = binary decision to assign a $\$10$ discount to customer $i$
- $x_{i, 20} \in \{0, 1\}$ = binary decision to assign a $\$20$ discount to customer $i$
- $\tau_{i, 10}$ = estimated ITE of $\$10$ discount (lift in conversion probability)
- $\tau_{i, 20}$ = estimated ITE of $\$20$ discount (lift in conversion probability)
- $c_{10}, c_{20}$ = cost per discount ($10 and $20, respectively)
- $B$ = total marketing budget constraint

$$\max_{\{x_{i, 10}, x_{i, 20}\}} \sum_{i=1}^N \left( \tau_{i, 10} x_{i, 10} + \tau_{i, 20} x_{i, 20} \right)$$

$$\text{subject to:}$$
1. **Budget Constraint:**
$$\sum_{i=1}^N \left( c_{10} x_{i, 10} + c_{20} x_{i, 20} \right) \le B$$

2. **Exclusivity Constraint (At most one promotion per customer):**
$$x_{i, 10} + x_{i, 20} \le 1, \quad \forall i \in \{1, \dots, N\}$$

3. **Integrality Constraint:**
$$x_{i, 10}, x_{i, 20} \in \{0, 1\}, \quad \forall i \in \{1, \dots, N\}$$


In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import csc_matrix, eye, hstack, vstack

# Set aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 5)
print("Libraries loaded successfully!")


Libraries loaded successfully!


### 1. Load Individual Treatment Effect (ITE) Predictions
We load the customer-level ITE predictions generated by EconML Double ML in Week 2 (`data/ite_predictions.csv`).
Each record contains the predicted incremental conversion lift $\tau_{i, 10}$ and $\tau_{i, 20}$.


In [2]:
df_ite = pd.read_csv('data/ite_predictions.csv')
print(f"Loaded {len(df_ite):,} customer records.")
df_ite[['client_id', 'age', 'has_redeemed', 'ite_10', 'ite_20']].head()


Loaded 10,000 customer records.


,client_id,age,has_redeemed,ite_10,ite_20
0,582caf677d,43,0,-0.007027,0.068457
1,7345cbbfdb,72,1,0.043879,0.045112
2,ba7e16dc51,43,1,0.016555,0.048884
3,f47784cc95,38,1,0.031525,0.047295
4,02a400777e,53,1,0.030411,0.047079
